# 06 — Edge: export, nicemleme, doğruluk (E9)
**H5'te eğitim yok:** eğitilmiş `s` modelleri (E1, E2, E3c) dışa aktarılır ve ölçülür.
- `PART = 'onnx'` → **Çalışma zamanı → A100** (GPU kullanılmaz; 12 CPU çekirdeği ORT CPU
  doğruluk ölçümünü hızlandırır). ONNX FP32/FP16/INT8/INT8-head-FP32 + laptop zip'i.
- `PART = 'trt'` → **Çalışma zamanı → T4** (TensorRT motoru donanıma özgü). FP32/FP16/INT8,
  doğruluk ve T4 gecikmesi.

Secrets: `KAGGLE_API_TOKEN`, `GITHUB_TOKEN`. **Kullanım:** ilk hücrede `PART`'ı seç →
**Tümünü çalıştır**. Biten adımlar atlanır; oturum koparsa yeniden Tümünü çalıştır.
İş bitince oturumu hemen sonlandır.

In [ ]:
BRANCH = 'h5'
PART = 'onnx'   # 'onnx' (A100 çalışma zamanı) | 'trt' (T4)
DRIVE = '/content/drive/MyDrive/baykus'
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
print('token yüklendi (değer yazdırılmadı)')

In [ ]:
# Repo private ise Secrets'a GITHUB_TOKEN ekle (fine-grained, sadece bu repo, 'Contents: read').
import os
from google.colab import userdata
try:
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = f'https://{tok}@github.com/OgzhnTarhn/baykus.git' if tok else 'https://github.com/OgzhnTarhn/baykus.git'
if not os.path.isdir('/content/baykus/.git'):
    !git clone -q {url} /content/baykus
!git -C /content/baykus fetch -q {url} {BRANCH}
!git -C /content/baykus checkout -q -B {BRANCH} FETCH_HEAD
!git -C /content/baykus remote set-url origin https://github.com/OgzhnTarhn/baykus.git
%cd /content/baykus
!git log --oneline -3

In [ ]:
%%time
# Kurulum (≈ 1 dk). Sürüm satırlarını Claude'a getir.
!pip -q install uv kaggle
if PART == 'trt':
    !pip -q install tensorrt
!uv pip install --system -q -e '/content/baykus/python[train]'
import sys
sys.path.insert(0, '/content/baykus/python/src')
!python --version && nproc && (nvidia-smi -L || true)
!python -c "import onnxruntime as o, ultralytics as u; print('onnxruntime', o.__version__, '· ultralytics', u.__version__)"
!python -c "import tensorrt as t; print('tensorrt', t.__version__)" || true

In [ ]:
%%time
# İşlenmiş veri: private Kaggle veri seti (ADR 0004). Kaggle yüklenen tar'ları klasöre açtı;
# CLI klasör seçerek indiremediği için veri setinin tamamı iner (~9,8 GB).
DS = 'ouzhantarhan/baykus-dv-rgbt'
!kaggle datasets download {DS} -p /content/dv_rgbt --unzip -q
!mv /content/dv_rgbt/meta /content/meta
!baykus-data illum-relabel --meta /content/meta   # ADR 0005: Kaggle'daki etiketler eski eşikle
!baykus-data slices --out /content/dv_rgbt --meta /content/meta
!baykus-data yaml --out /content/dv_rgbt
!du -sh /content/dv_rgbt/* /content/meta

In [ ]:
%%time
# Modeller ve kalibrasyon listesi (≈ 1-5 dk; T4'ün 2 çekirdeğinde rgbt4 üretimi daha uzun).
import json, os
from pathlib import Path
import yaml
from baykus.train.config import load_config
NPROC = os.cpu_count()
REPO = Path('/content/baykus')
E = yaml.safe_load((REPO / 'configs/edge/e9_edge.yaml').read_text(encoding='utf-8'))
RUN = {k: v['run_id'] for k, v in E['models'].items()}
MOD = {k: load_config(REPO / v['config']).modality for k, v in E['models'].items()}
W = {k: Path(f'{DRIVE}/runs/{r}/weights/best.pt') for k, r in RUN.items()}
missing = [str(p) for p in W.values() if not p.exists()]
assert not missing, f'best.pt yok: {missing}'
MODELS_DIR = f'{DRIVE}/edge/models'
PROJ = f'{DRIVE}/edge/eval'
CFG = '../configs/edge/e9_edge.yaml'
for m in sorted(set(MOD.values())):
    !cd /content/baykus/python && baykus-edge calib --config {CFG} --data-root /content/dv_rgbt --modality {m} --out /content/calib --build-rgbt4 --workers {NPROC}
if 'rgbt4' in MOD.values():
    !baykus-data rgbt4 --out /content/dv_rgbt --splits val --workers {NPROC}
VARIANTS = ['onnx_fp32', 'onnx_fp16', 'onnx_int8', 'onnx_int8_headfp32'] if PART == 'onnx' else ['trt_fp32', 'trt_fp16', 'trt_int8']
EXT = '.onnx' if PART == 'onnx' else '.engine'
for k in RUN:
    print(k, RUN[k], MOD[k])

In [ ]:
%%time
# Export (onnx: model başına ≈ 1-3 dk, INT8 kalibrasyonu dahil; trt: motor başına ≈ 3-6 dk).
# Biten varyant manifest'ten atlanır. Wall time'ı ve son 20 satırı Claude'a getir.
for k in RUN:
    !cd /content/baykus/python && baykus-edge export --config {CFG} --model {k} --weights "{W[k]}" --out {MODELS_DIR} --variants {' '.join(VARIANTS)} --data-root /content/dv_rgbt --calib-dir /content/calib
!ls -la {MODELS_DIR}

In [ ]:
%%time
# Doğruluk (val, IR etiketleri, bootstrap yok; FR-X4). ONNX: ORT CPU, bütün çekirdekler;
# ilk varyantın süresine bakarak kalanı tahmin et. 'map50=' satırlarını Claude'a getir.
THREADS = f'--threads {NPROC}' if PART == 'onnx' else ''
for k in RUN:
    for V in VARIANTS:
        F = f'{MODELS_DIR}/{RUN[k]}_{V}{EXT}'
        !cd /content/baykus/python && baykus-edge eval --config {CFG} --model {k} --variant {V} --file "{F}" --data-root /content/dv_rgbt --meta /content/meta --project {PROJ} --results {DRIVE}/results {THREADS}
# E4: iki modelin aynı varyantı, H4'te val'de seçilen WBF parametreleriyle
for V in VARIANTS:
    R = f'{PROJ}/{RUN["e1_rgb"]}_val_{V}/predictions_coco.json.gz'
    I = f'{PROJ}/{RUN["e2_ir"]}_val_{V}/predictions_coco.json.gz'
    !cd /content/baykus/python && baykus-fusion apply --config ../configs/fusion/e4_late_fusion.yaml --selection ../results/fusion/e4_late_fusion_val_selection.json --rgb-preds "{R}" --ir-preds "{I}" --data-root /content/dv_rgbt --meta /content/meta --split val --dataset dronevehicle --variant {V} --n-boot 0 --project {PROJ} --results {DRIVE}/results --workers {NPROC}

In [ ]:
%%time
# Yalnız PART='onnx': laptop için modeller + bench girdileri tek zip'te (≈ 1 dk).
# 'du' satırlarını Claude'a getir; boyutu görmeden indirme.
if PART == 'onnx':
    !rm -rf /content/edge_laptop && mkdir -p /content/edge_laptop/models
    !cd /content/baykus/python && baykus-edge bench-inputs --config {CFG} --data-root /content/dv_rgbt --split val --preds-dir {PROJ} --out /content/edge_laptop/bench_inputs
    !cp {MODELS_DIR}/*.onnx /content/edge_laptop/models/
    !du -sh /content/edge_laptop/models /content/edge_laptop/bench_inputs
    !cd /content/edge_laptop && zip -q -r -0 {DRIVE}/edge/edge_laptop.zip models bench_inputs
    !du -sh {DRIVE}/edge/edge_laptop.zip
else:
    print("PART='trt' → atlandı")

In [ ]:
%%time
# Yalnız PART='trt': T4 gecikmesi (§14: 50 ısınma + 500 ölçüm; ölçüm başına ≈ 1 dk).
if PART == 'trt':
    !cd /content/baykus/python && baykus-edge bench-inputs --config {CFG} --data-root /content/dv_rgbt --split val --preds-dir {PROJ} --out /content/bench_inputs --variants {' '.join(VARIANTS)}
    for P in E['pipelines']:
        for V in VARIANTS:
            !cd /content/baykus/python && baykus-bench run --config {CFG} --pipeline {P} --variant {V} --models-dir {MODELS_DIR} --inputs /content/bench_inputs --hardware colab-t4 --results {DRIVE}/results
else:
    print("PART='onnx' → atlandı (laptop bench'i Görev 14)")